In [1]:
#Klasyfikatory
from capymoa.classifier import EFDT
from capymoa.classifier import AdaptiveRandomForestClassifier
from capymoa.classifier import LeveragingBagging
from capymoa.classifier import StreamingRandomPatches
from capymoa.classifier import StreamingGradientBoostedTrees
#AdaptiveRandomForestClassifier
#StreamingGradientBoostedTrees
#StreamingRandomPatches
#LeveragingBagging
from capymoa.classifier import OnlineBagging
from capymoa.classifier import OzaBoost
models_classes = [OnlineBagging, OzaBoost, EFDT]

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [3]:
from capymoa.datasets import ElectricityTiny
from capymoa.datasets import CovtypeNorm
from capymoa.stream import ARFFStream

#plik z https://www.openml.org/search?type=data&sort=version&status=any&order=asc&exact_name=airlines&id=45072
stream_air = ARFFStream('data/airline.arff')
stream_air.restart()
stream_cover = CovtypeNorm()
stream_electric = ElectricityTiny()

In [4]:
from capymoa.evaluation import ClassificationWindowedEvaluator


In [8]:
def test(models_class, stream, filename="plik.csv"):
    stream.restart()
    window_size = 200
    evaluator = ClassificationWindowedEvaluator(stream.get_schema(), window_size=window_size)

    N=0
    for i in enumerate(stream):
        N += 1

    mx = int(N/window_size)
    ny = [ i*window_size for i in range(1,mx)]
    if N%window_size:
      ny.append(N)  
    ny = np.array(ny, dtype=int)

    models = [m(stream.get_schema()) for m in models_class]
    models_name = [model.__class__.__name__ for model in models]
    
    results = pd.DataFrame(np.nan, index=ny, columns=models_name)
    
    for model, model_name in zip(models, models_name):
        stream.restart()
        acc = []
        for idi, instance in enumerate(stream):
    
            y_pred = model.predict(instance)
                    
            evaluator.update(y_target_index=instance.y_index, y_pred_index=y_pred)
        
            model.train(instance)

            if (idi + 1) in ny:
                acc.append(evaluator.accuracy()[-1])  

            if (idi + 1)%10000 == 0 :
                print((idi+1)/N)
        print(len(ny), len(acc))
        results[model_name] = acc
    results.to_csv(filename)
    return results
        

In [10]:
streams = [stream_air,stream_cover,stream_electric]
streams_names = ["airline.csv", "covertypeNorm.csv", "electricity_tiny.csv"]
for stream, filename in zip(streams, streams_names):
    test(models_classes, stream, filename)


0.018539701844514936
0.03707940368902987
0.05561910553354481
0.07415880737805974
0.09269850922257468
0.11123821106708962
0.12977791291160456
0.1483176147561195
0.16685731660063444
0.18539701844514936
0.2039367202896643
0.22247642213417923
0.24101612397869418
0.25955582582320913
0.27809552766772405
0.296635229512239
0.3151749313567539
0.3337146332012689
0.3522543350457838
0.3707940368902987
0.38933373873481364
0.4078734405793286
0.42641314242384354
0.44495284426835846
0.46349254611287344
0.48203224795738836
0.5005719498019033
0.5191116516464183
0.5376513534909332
0.5561910553354481
0.574730757179963
0.593270459024478
0.6118101608689929
0.6303498627135078
0.6488895645580228
0.6674292664025377
0.6859689682470527
0.7045086700915676
0.7230483719360825
0.7415880737805974
0.7601277756251124
0.7786674774696273
0.7972071793141423
0.8157468811586572
0.8342865830031722
0.8528262848476871
0.871365986692202
0.8899056885367169
0.9084453903812318
0.9269850922257469
0.9455247940702618
0.96406449591477

40200


0.09269850922257468


54800
55200
55400
56600
59200
59400
59800
60000


0.11123821106708962


...

0.9638355145848967


561000
562600
563200


0.981046863059627


572600
575600
575600
575800
578200
578200
578400


0.9982582115343573
2905 2905
9 9
9 9
9 9
